# Fine-Tuning, Model Routing, and Prompt Optimization

Custom models across GPT, Claude, Gemini, Mistral, DeepSeek, and Kimi — tuned for cost, latency, and quality.

This notebook is an original tutorial extending the course with current
(2026) state-of-the-art practice for the model layer of a production stack:
when to fine-tune, which open-weight models matter, how routing works, and
how prompts get optimized systematically instead of by hand.

## Learning Objectives

- Choose between SFT, DPO, and GRPO, and explain LoRA/QLoRA mechanics and why they're cheap.
- State when fine-tuning wins (style, format, cost arbitrage) and when it loses (knowledge — RAG wins).
- Know the open-weight landscape (DeepSeek, Kimi, Qwen, Mistral, Llama) and why a router includes them.
- Compare learned routing (RouteLLM), cascade routing, and capability-flag routing.
- Describe systematic prompt optimization: DSPy signatures/optimizers, GEPA, prompt compression, meta-prompting.

## Mental Model

Three levers move the cost/latency/quality triangle, in order of increasing
commitment:

1. **Prompt optimization** — hours, reversible, no infra. Try first.
2. **Routing** — days of engineering; send each query to the cheapest model
   that can handle it. RouteLLM's headline: 85% cost reduction at 95% of
   GPT-4 quality (strong model needed on only 14% of queries).
3. **Fine-tuning** — weeks, needs eval discipline; bakes behavior into a
   smaller model so you can route *more* traffic to the cheap tier.

They compose: fine-tune a small model on the frontier model's outputs
(distillation), route most traffic to it, escalate on low confidence, and
optimize the prompts per model — because **prompts do not transfer across
models**.

## Important Concepts

- **SFT**: input→output pairs; teaches format, style, tool-calling. Always the first stage.
- **DPO**: preference pairs, no reward model — "the first thing after SFT".
- **GRPO** (DeepSeek-R1's method): group of completions per prompt, rewards normalized within group, no critic — the standard for verifiable-reward tasks (math, code, structured output).
- **LoRA/QLoRA**: freeze base weights, train low-rank update ΔW = B·A (rank 8-64) → ~0.1-1% trainable params; QLoRA quantizes the base to 4-bit → 70B tunable on one GPU. Adapters are MB-sized, hot-swappable (multi-tenant serving via vLLM/LoRAX).
- **Distillation**: frontier model generates training data for a small model; first-class provider workflows (OpenAI stored completions → FT dataset; Gemini Distillation Service). ToS caveat: distilling into a *competing* model violates terms.
- **Routing types**: learned (RouteLLM-style classifier), cascade (cheap first, escalate on low confidence), capability-flags/rules (most production "routing" is this plus fallbacks).
- **DSPy**: signatures declare input→output contracts; optimizers (MIPROv2, GEPA) compile prompts against a metric — prompts become optimized artifacts, not hand-written strings.

## Need-To-Know Coverage Checklist

- [x] SFT → DPO → GRPO decision rule; RLHF/PPO confined to frontier labs.
- [x] LoRA rank/adapter mechanics; QLoRA; multi-adapter serving.
- [x] Provider offerings: OpenAI (SFT/DPO + RFT with programmatic graders), Gemini tuning + distillation, Mistral; **Anthropic has no self-serve fine-tuning** (Bedrock Haiku only) — a classic interview trap.
- [x] Fine-tune vs RAG: knowledge injection → RAG wins, decisively.
- [x] Dataset sizes (50-100 floor, 200-500 sweet spot, >5000 rarely helps) and eval-before/after discipline.
- [x] Open-weight models: DeepSeek (MoE cost arbitrage), Kimi K2 (agentic/long-context), Qwen (size ladder, fine-tuning substrate), Mistral (EU residency), Llama (ecosystem).
- [x] RouteLLM numbers; cascade routing; OpenRouter/NotDiamond/LiteLLM routing strategies.
- [x] DSPy, MIPROv2, GEPA (beats GRPO by up to 20% with 35x fewer rollouts); LLMLingua compression; Anthropic prompt improver.

## Deep Study Notes

### Fine-tuning: methods and mechanics

- Decision rule: **SFT first** (format/style/tool-calling), **DPO** if you
  have preference pairs, **GRPO** when the reward is programmatically
  verifiable (does it compile, does the JSON validate, is the answer
  numerically correct).
- **Why RLHF/PPO stays in frontier labs**: it requires training and hosting a
  separate reward model, plus a notoriously unstable PPO loop with a critic/
  value model — multiple large models in memory and heavy tuning. DPO removes
  the reward model; GRPO removes the critic (rewards normalized within a
  sampled group). That's why product teams stop at DPO/GRPO.
- **LoRA**: ΔW = B·A with rank r«d cuts trainable parameters to ~0.1-1%,
  so one GPU suffices and the artifact is a hot-swappable MB-scale adapter —
  which enables the standard multi-tenant pattern: one base model, N adapters
  routed per request.
- Practical numbers: 50-100 examples is the floor for measurable gains,
  200-500 hand-curated is the sweet spot, >5000 rarely helps — quality over
  quantity. A hosted SFT job on a mini model runs $5-50.
- **Provider landscape**: OpenAI (SFT/DPO, plus Reinforcement Fine-Tuning on
  reasoning models where *you supply a programmatic grader*), Google (Vertex
  tuning + a distillation service), Mistral (tune anywhere — open weights).
  **Anthropic offers no self-serve fine-tuning** (only Haiku SFT via Bedrock);
  their position is that strong prompting + tools beats FT for most uses.
  If someone says "we fine-tuned Claude", the correct follow-up is "via
  Bedrock, or do you mean prompt optimization?"

### Fine-tune vs RAG vs prompt (the decision interviewers test)

- FT **wins**: persona/tone consistency, strict output formats, narrow
  repetitive tasks (classification, extraction, routing itself), tool-call
  reliability, and cost/latency arbitrage — a tuned small model replacing a
  frontier model at 10-30x lower cost. It also shrinks prompts (behavior
  baked in).
- FT **loses**: knowledge injection. Facts, freshness, citations → **RAG
  wins decisively**; FT doesn't reliably store new knowledge and can't be
  updated per-document.
- The production pattern: RAG for knowledge + a fine-tuned small model for
  behavior/format on top.
- Discipline that separates credible teams: held-out eval built *before*
  training, baseline the prompted frontier model, identical evals
  before/after, regression-check for capability loss, ship behind an A/B.
  "We fine-tuned and it felt better" is the anti-pattern.

### Open-weight models and why routers include them

| Model | Known for | Routing rationale |
|---|---|---|
| DeepSeek V3.x/R1 | sparse MoE (~671B total/37B active), MIT license, GRPO-trained reasoning | frontier-adjacent quality at ~1/10-1/50 the price |
| Kimi K2.x (Moonshot) | 1T/32B-active MoE, best-in-class agentic coding, long context | route agentic/long-context workloads cheaply |
| Qwen 3.x (Alibaba) | >50% of open-model downloads; size ladder 0.6B→480B | distillation targets, edge/cheap tiers, best FT substrate |
| Mistral | European — EU data residency, Apache-2 small models | compliance-driven routing, on-prem |
| Llama 4 | open multimodal MoE, ecosystem gravity | self-hosting default, tooling |

Cross-cutting: cost arbitrage (40-85% savings), data residency (nothing
leaves the VPC), fine-tunability (you can't LoRA GPT-5), no vendor
rate-limit coupling.

### Model routing

- **Learned (RouteLLM-style)**: a classifier trained on preference data
  predicts whether the cheap model suffices. Headline result: 85% cost
  reduction on MT-Bench at 95% of GPT-4 quality; the strong model was needed
  on only 14% of queries.
- **Cascade**: try cheap, escalate on low confidence (logprobs, self-reported
  confidence, or judge verification). No training needed; costs latency on
  escalated queries.
- **Capability-flags/rules**: long context → Kimi/Gemini; vision → multimodal;
  PII/regulated → EU-hosted Mistral; function calling → whichever model your
  evals show is reliable. Most production "routing" is this plus fallbacks.
- Tools: OpenRouter (auto-router with a cost/quality dial), NotDiamond
  (trainable on your evals), Martian, LiteLLM (latency-, cost-, usage-based
  strategies + budgets + fallbacks).
- The honest framing: basic routing is *days of engineering, not research*.
  The hard parts are the eval harness for routing decisions, the confidence
  signal, failover behavior, and per-provider prompt adaptation.

### Systematic prompt optimization

- **DSPy**: declare *signatures* (input→output contracts); modules (Predict,
  ChainOfThought, ReAct) compile against a metric. Optimizers: BootstrapFewShot
  (self-generate demos), **MIPROv2** (jointly optimizes instructions + demos
  via Bayesian search).
- **GEPA** (Genetic-Pareto, ICLR 2026): reflective prompt evolution — natural-
  language reflection on execution traces + evolutionary search + Pareto
  candidate selection. Beats MIPROv2 by ~10-13% and **GRPO by up to 20% with
  35x fewer rollouts** — prompt optimization beating RL fine-tuning at a
  fraction of the cost. In production at e.g. Decagon for classification/
  agent prompts.
- **Prompt compression (LLMLingua)**: small LM prunes low-information tokens;
  4-10x typical in production with ~1.5pt accuracy loss; layered under
  provider prompt caching (stable prefixes cached, dynamic context compressed).
- **Meta-prompting**: a strong model writes/critiques prompts for a cheaper
  one — this is what Anthropic's Console prompt improver does (~30% accuracy
  boost on a classification test) and how routers adapt prompts per provider.
- Production reality: most teams do eval-driven iteration (versioned prompt
  registry, golden set, CI gates, A/B rollout); DSPy/GEPA adoption
  concentrates in teams with strong eval infrastructure, because every
  optimizer is only as good as its metric.

## Common Failure Modes

- Fine-tuning to inject knowledge → stale, uncitable, per-document updates impossible; should have been RAG.
- No held-out eval before training → cannot prove the tune helped; capability regressions ship silently.
- One prompt shared across six providers → the router "works" but quality silently varies per model.
- Cascade router with no confidence calibration → escalates everything (no savings) or nothing (quality cliff).
- Optimizing prompts against a weak metric → GEPA/DSPy faithfully overfit to the wrong thing.
- Claiming "fine-tuned Claude" without knowing it's Bedrock-only → credibility hit in any technical interview.

## Implementation Notes

- Order of operations: prompt optimization → routing → fine-tuning. Each step needs the eval suite the previous one built.
- Keep a per-model prompt variant registry; route + prompt version together as one deployable unit.
- For verifiable tasks, write the grader first — it serves evals, GRPO/RFT, and cascade confidence all at once.
- Distill deliberately: log frontier-model traffic, curate 200-500 gold examples, tune the small model, A/B it.

In [ ]:
"""Cascade routing with a confidence signal, plus the cost math.

The core production routing pattern: try the cheap model, escalate when
confidence is low, and measure the cost/quality tradeoff.
"""
QUERIES = [  # (query, difficulty 0-1) — difficulty unknown to the router
    ("What's your refund window?", 0.1),
    ("Summarize this 40-page contract's indemnity risks", 0.9),
    ("Translate 'hello' to French", 0.05),
    ("Reconcile these three conflicting API specs", 0.8),
    ("What are your support hours?", 0.1),
]

PRICE = {"small": 0.4, "frontier": 8.0}  # $ per M tokens (illustrative)


def small_model(query, difficulty):
    """Cheap model: good on easy queries; confidence tracks difficulty."""
    confidence = 1.0 - difficulty
    quality = 0.95 if difficulty < 0.5 else 0.55
    return {"quality": quality, "confidence": confidence}


def cascade(query, difficulty, threshold=0.6):
    result = small_model(query, difficulty)
    if result["confidence"] >= threshold:
        return {"model": "small", "quality": result["quality"]}
    return {"model": "frontier", "quality": 0.97}  # escalate


routed = [cascade(q, d) for q, d in QUERIES]
n_small = sum(1 for r in routed if r["model"] == "small")
cost = n_small * PRICE["small"] + (len(routed) - n_small) * PRICE["frontier"]
cost_all_frontier = len(routed) * PRICE["frontier"]
avg_q = sum(r["quality"] for r in routed) / len(routed)

for (q, _), r in zip(QUERIES, routed):
    print(f"  [{r['model']:>8}] q={r['quality']:.2f}  {q[:50]}")
print(f"\ncost: ${cost:.1f} vs all-frontier ${cost_all_frontier:.1f} "
      f"({(1 - cost / cost_all_frontier):.0%} saved) at avg quality {avg_q:.2f}")
# The whole game is the confidence threshold: sweep it and you trace the
# cost/quality Pareto curve — the operating point is a product decision.


## Practice

1. Sweep `threshold` from 0.1 to 0.9 and print the cost/quality pairs — you've
   just drawn the routing Pareto curve. Where would a support bot vs a legal
   tool sit on it?
2. The small model is *miscalibrated*: make its confidence always 0.9. What
   happens to quality, and which fix from the notes applies?
3. You have 300 gold examples of perfectly-formatted extraction outputs from
   a frontier model. Walk the distillation play: which method (SFT/DPO/GRPO),
   which target model class, what eval discipline before shipping.
4. A stakeholder wants to "fine-tune the model on our product docs so it
   knows our product." Give the two-sentence correction and the architecture
   you'd propose instead.
5. Explain GEPA's headline result (vs GRPO, 35x fewer rollouts) and why it
   reorders the "prompt-opt before fine-tuning" advice even for hard tasks.

## Design Checklist

- [ ] Eval suite exists before any tuning or routing decision.
- [ ] Fine-tuning reserved for behavior/format/cost — never knowledge.
- [ ] Held-out before/after evals with capability-regression checks.
- [ ] Router has a measured confidence signal and a Pareto operating point chosen deliberately.
- [ ] Per-model prompt variants, versioned; prompt + model deploy as one unit.
- [ ] Fallback chain across providers; budgets and rate limits at the gateway.
- [ ] Distillation data curated (200-500 gold), not dumped.
- [ ] Prompt optimization is metric-driven (DSPy/GEPA or eval-gated iteration), not vibes.